### **Step 1 — Import libraries**


In [26]:
import os
import numpy as np
import lightgbm as lgb
import joblib
from sklearn.metrics import accuracy_score, classification_report


### **STEP 2 — Define dataset paths**

In [27]:
BASE = r"D:\Deepfake_Audio_Project\features\mfcc"
MODEL_PATH = r"D:\Deepfake_Audio_Project\models"

os.makedirs(MODEL_PATH, exist_ok=True)

print("MFCC path:", BASE)
print("Model save path:", MODEL_PATH)


MFCC path: D:\Deepfake_Audio_Project\features\mfcc
Model save path: D:\Deepfake_Audio_Project\models


### **STEP 3 — Load TRAIN and DEV data**

In [28]:
# TRAIN
X_train = np.load(BASE + r"\X_train.npy")
y_train = np.load(BASE + r"\y_train.npy")

# DEV
X_dev = np.load(BASE + r"\X_dev.npy")
y_dev = np.load(BASE + r"\y_dev.npy")

print("Train:", X_train.shape, y_train.shape)
print("Dev:", X_dev.shape, y_dev.shape)


Train: (25380, 20) (25380,)
Dev: (24844, 20) (24844,)


### **Step 4 — Create LightGBM model**

In [29]:
model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=64,
    max_depth=-1,
    random_state=42
)

print("Model created")


Model created


### **Step 5 — Train the model**

In [30]:
model.fit(X_train, y_train)
print("Training finished")


[LightGBM] [Info] Number of positive: 22800, number of negative: 2580
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000979 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5100
[LightGBM] [Info] Number of data points in the train set: 25380, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.898345 -> initscore=2.178971
[LightGBM] [Info] Start training from score 2.178971
Training finished


### **Step 6 — Validate on DEV set**

In [31]:
dev_pred = model.predict(X_dev)

print("DEV Accuracy:", accuracy_score(y_dev, dev_pred))
print(classification_report(y_dev, dev_pred))


c:\Users\purud\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


DEV Accuracy: 0.9203026887779746
              precision    recall  f1-score   support

           0       0.75      0.34      0.46      2548
           1       0.93      0.99      0.96     22296

    accuracy                           0.92     24844
   macro avg       0.84      0.66      0.71     24844
weighted avg       0.91      0.92      0.91     24844



### **STEP 7 — Save trained model**

In [32]:
joblib.dump(model, MODEL_PATH + r"\lightgbm_mfcc.pkl")
print("LightGBM model saved")


LightGBM model saved


### **STEP 8 — Get DEV probabilities**

In [33]:
# probability that sample is FAKE
dev_probs = model.predict_proba(X_dev)[:, 1]

print("Example probabilities:", dev_probs[:10])


Example probabilities: [0.93144258 0.62190144 0.99800335 0.99952791 0.28294489 0.90883355
 0.98824143 0.99556766 0.93011283 0.99875743]


c:\Users\purud\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### **STEP 9 — Save DEV probabilities**

In [34]:
np.save(MODEL_PATH + r"\lgbm_dev_probs.npy", dev_probs)
np.save(MODEL_PATH + r"\dev_labels.npy", y_dev)

print("DEV probabilities saved")


DEV probabilities saved


### **STEP 10 — Load EVAL data**

In [35]:
X_eval = np.load(BASE + r"\X_eval.npy")
y_eval = np.load(BASE + r"\y_eval.npy")

print("Eval:", X_eval.shape, y_eval.shape)


Eval: (71237, 20) (71237,)


### **STEP 11 — Generate EVAL probabilities**

In [36]:
eval_probs = model.predict_proba(X_eval)[:, 1]

print("Eval probabilities sample:", eval_probs[:10])


c:\Users\purud\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Eval probabilities sample: [0.02387863 0.11623166 0.09446295 0.56743237 0.95660333 0.2534573
 0.88864727 0.97330726 0.03293729 0.99409014]


### **STEP 12 — Save EVAL probabilities**

In [37]:
np.save(MODEL_PATH + r"\lgbm_eval_probs.npy", eval_probs)
np.save(MODEL_PATH + r"\eval_labels.npy", y_eval)

print("EVAL probabilities saved")


EVAL probabilities saved


In [ ]:
import os
import numpy as np
import lightgbm as lgb
import joblib
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Paths
BASE = r"D:\Deepfake_Audio_Project\features\mfcc"
MODEL_PATH = r"D:\Deepfake_Audio_Project\models"

# Load the saved model
print("Loading saved LightGBM model...")
model = joblib.load(MODEL_PATH + r"\lightgbm_mfcc.pkl")
print("✅ Model loaded!")

# Load EVAL data
X_eval = np.load(BASE + r"\X_eval.npy")
y_eval = np.load(BASE + r"\y_eval.npy")
print(f"EVAL data: {X_eval.shape}")

# ===== EVAL CHECK =====
print("\n" + "="*60)
print("CHECKING EVAL PERFORMANCE")
print("="*60)

eval_pred = model.predict(X_eval)

print("\n📊 EVAL Accuracy:", accuracy_score(y_eval, eval_pred))

print("\n📊 EVAL Classification Report:")
print(classification_report(y_eval, eval_pred, target_names=['Real', 'Fake']))

print("\n📊 EVAL Confusion Matrix:")
cm = confusion_matrix(y_eval, eval_pred)
print(cm)
print("Format: [[True_Real, False_Fake],")
print("         [False_Real, True_Fake]]")

tn, fp, fn, tp = cm.ravel()
real_recall = tn / (tn + fp) if (tn + fp) > 0 else 0
fake_recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"\n🎯 EVAL Per-Class Recall:")
print(f"Real recall: {real_recall:.4f}")
print(f"Fake recall: {fake_recall:.4f}")

if real_recall > 0.30 and fake_recall > 0.95:
    print("\n✅ LGBM LOOKS GOOD! Both classes detected.")
else:
    print("\n⚠️  LGBM might be biased")

print("\n" + "="*60)

Loading saved LightGBM model...
✅ Model loaded!
EVAL data: (71237, 20)

CHECKING EVAL PERFORMANCE


d:\Deepfake_Audio_Project\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



📊 EVAL Accuracy: 0.8854387467186996

📊 EVAL Classification Report:
              precision    recall  f1-score   support

        Real       0.45      0.54      0.50      7355
        Fake       0.95      0.92      0.94     63882

    accuracy                           0.89     71237
   macro avg       0.70      0.73      0.72     71237
weighted avg       0.90      0.89      0.89     71237


📊 EVAL Confusion Matrix:
[[ 4003  3352]
 [ 4809 59073]]
Format: [[True_Real, False_Fake],
         [False_Real, True_Fake]]

🎯 EVAL Per-Class Recall:
Real recall: 0.5443
Fake recall: 0.9247

⚠️  LGBM might be biased

